## librerias

In [16]:
import os
import webbrowser
import pandas as pd
import json
import geopandas as gpd
import colorsys
import numpy as np
import http.server
import socketserver
from threading import Thread
import time
import hashlib
import unicodedata

## bases 

In [17]:
usuario = os.getlogin()

In [18]:
base = pd.read_excel(fr"C:\Users\{usuario}\Downloads\UnidadesIMB_CS!_v2.xlsx",sheet_name="Sheet 1")
clues = pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Repositorio de Datos\CLUES\clues.parquet")

In [19]:
base.columns

Index(['clues_imb', 'nombre_de_la_unidad', 'categoria_gerencial_ampliada',
       'poblacion_sin_dh_menos_30', 'poblacion_con_dh_menos_30', 'pob_imo_pct',
       'CONSULTORIOS GENERALES', 'consultorios_generales_habilitados',
       'Equipo de cómputo', 'Banco de altura', 'Banco giratorio',
       'Báscula electrónica  con estadímetro',
       'Báscula pesabebés electrónica ', 'Bote sanitario con pedal',
       'Caja Portalaminilla de plástico con separadores',
       'Carta Snellen con marco',
       'Charola de Mayo de acero inoxidable, Dimensiones: 49x32cm',
       'Cinta métrica', 'Contenedor de jabón líquido',
       'Contenedor de toallas desechables',
       'Contenedor rígido 7.50 a 9.40 ml.',
       'Cubeta de Acero Inoxidable y bolsa', 'Equipo de cómputo.1',
       'Escritorio médico de 150x60x75', 'Esfigmomanómetro ',
       'Espejo vestidor',
       'Espejos graves o vaginales chicos, medianos y grandes',
       'Estadímetro pediátrico',
       'Estetoscopio cápsula doble. 

## back

In [20]:
base = base.drop(columns=['poblacion_sin_dh_menos_30', 'poblacion_con_dh_menos_30', 'pob_imo_pct'])


base.columns = (
    base.columns
        .str.strip()
        .str.lower()
        .map(
            lambda x: ''.join(
                c for c in unicodedata.normalize('NFD', x)
                if unicodedata.category(c) != 'Mn'
            )
        )
        .str.replace(" ", "_", regex=False)
)

In [21]:
base = base.merge(
    clues[["clues_imb", "entidad"]],
    on="clues_imb",
    how="left"
)

In [22]:
base =  base.drop(columns=['categoria_gerencial_ampliada','consultorios_generales', 'consultorios_generales_habilitados'])

In [23]:
base.columns

Index(['clues_imb', 'nombre_de_la_unidad', 'equipo_de_computo',
       'banco_de_altura', 'banco_giratorio',
       'bascula_electronica__con_estadimetro', 'bascula_pesabebes_electronica',
       'bote_sanitario_con_pedal',
       'caja_portalaminilla_de_plastico_con_separadores',
       'carta_snellen_con_marco',
       'charola_de_mayo_de_acero_inoxidable,_dimensiones:_49x32cm',
       'cinta_metrica', 'contenedor_de_jabon_liquido',
       'contenedor_de_toallas_desechables',
       'contenedor_rigido_7.50_a_9.40_ml.',
       'cubeta_de_acero_inoxidable_y_bolsa', 'equipo_de_computo.1',
       'escritorio_medico_de_150x60x75', 'esfigmomanometro', 'espejo_vestidor',
       'espejos_graves_o_vaginales_chicos,_medianos_y_grandes',
       'estadimetro_pediatrico',
       'estetoscopio_capsula_doble._auxiliar_para_realizar_auscultacion',
       'estetoscopio_pinard_o_doppler_fetal_portatil',
       'guarda_de_medicamentos,_materiales_o_instrumental',
       'guardarropa_con_perchero', 'lam

In [24]:
# Obtener entidades únicas
entidades_unicas = sorted(base['entidad'].dropna().unique())

# Identificar columnas de equipamiento (todas excepto las que no son numéricas)
columnas_excluir = ['clues_imb', 'nombre_de_la_unidad', 'categoria_gerencial_ampliada', 'entidad']
columnas_equipamiento = [col for col in base.columns if col not in columnas_excluir]

# Obtener unidades por entidad
def get_unidades_por_entidad(entidad):
    """Obtiene todas las unidades de una entidad con sus datos"""
    df_entidad = base[base['entidad'] == entidad]
    return df_entidad.to_dict('records')

In [25]:
# Configuración de colores
COLOR_PRIMARIO = "#FAF2F5"
COLOR_SECUNDARIO = '#AE8640'
COLOR_HBC = "#FDFDFDC0"
COLOR_FONDO = "#235B4E"
COLOR_BORDE = '#7A1737'
COLOR_TEXTO = '#000000'

In [26]:
# Función simple para formatear nombres de columnas
def formatear_nombre(columna):
    nombre = columna.replace('_', ' ')
    nombre = nombre.split('.')[0]
    palabras = nombre.split()
    palabras = [p.capitalize() for p in palabras]
    return ' '.join(palabras)


## front

In [27]:
script_url = "https://script.google.com/macros/s/AKfycbz_jWYNTXewhT3zFW4YPWx08OPSOpUQbH6dEdPw_3-XR2GRmiKukPbPZgf2Vi8zboJz/exec"

In [28]:
# Obtener columnas de equipamiento - EXCLUYENDO Consultorios Generales
columnas_excluir = ['clues_imb', 'nombre_de_la_unidad', 'categoria_gerencial_ampliada', 'entidad', 
                    'Consultorios_Generales', 'Consultorios_Generales_Habilitados']
columnas_equipamiento = [col for col in base.columns if col not in columnas_excluir]

html_content = f'''<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Cuestionario de Equipamiento - IMSS Bienestar</title>
    <link href="https://fonts.googleapis.com/css2?family=League+Spartan:wght@400;500;600;700&display=swap" rel="stylesheet">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
            font-family: "League Spartan", sans-serif;
        }}

        body {{
            background: linear-gradient(135deg, {COLOR_PRIMARIO} 0%, #fff 100%);
            min-height: 100vh;
            padding: 20px;
        }}

        .container {{
            width: 100%;
            max-width: 1400px;
            margin: 0 auto;
        }}

        .menu-container {{
            position: fixed;
            top: 20px;
            right: 20px;
            z-index: 1001;
        }}

        .menu-btn {{
            background: {COLOR_FONDO};
            border: none;
            border-radius: 50%;
            width: 50px;
            height: 50px;
            cursor: pointer;
            display: flex;
            flex-direction: column;
            justify-content: center;
            align-items: center;
            gap: 6px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.2);
            transition: 0.3s;
        }}

        .menu-btn:hover {{
            background: {COLOR_SECUNDARIO};
            transform: scale(1.05);
        }}

        .menu-btn span {{
            width: 25px;
            height: 3px;
            background: white;
            border-radius: 3px;
            transition: 0.3s;
        }}

        .menu-panel {{
            position: fixed;
            top: 0;
            right: -400px;
            width: 380px;
            height: 100%;
            background: white;
            box-shadow: -2px 0 10px rgba(0,0,0,0.1);
            z-index: 1002;
            transition: 0.3s;
            overflow-y: auto;
            padding: 80px 25px 25px 25px;
        }}

        .menu-panel.active {{
            right: 0;
        }}

        .menu-overlay {{
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0,0,0,0.5);
            z-index: 1001;
            display: none;
        }}

        .menu-overlay.active {{
            display: block;
        }}

        .menu-panel h2 {{
            color: {COLOR_FONDO};
            margin-bottom: 20px;
            font-size: 24px;
            border-bottom: 3px solid {COLOR_SECUNDARIO};
            padding-bottom: 10px;
        }}

        .menu-panel h3 {{
            color: {COLOR_SECUNDARIO};
            margin: 20px 0 10px 0;
            font-size: 18px;
        }}

        .menu-panel p {{
            color: #333;
            line-height: 1.6;
            margin-bottom: 15px;
        }}

        .menu-panel ul, .menu-panel ol {{
            color: #555;
            margin-left: 20px;
            margin-bottom: 15px;
        }}

        .menu-panel li {{
            margin-bottom: 8px;
        }}

        .close-menu {{
            position: absolute;
            top: 20px;
            right: 20px;
            background: none;
            border: none;
            font-size: 30px;
            cursor: pointer;
            color: {COLOR_FONDO};
        }}

        .header {{
            background: {COLOR_FONDO};
            padding: 20px;
            color: white;
            margin-bottom: 30px;
            border-radius: 15px;
            text-align: center;
            position: relative;
        }}

        .header img {{
            height: 50px;
            margin-bottom: 10px;
        }}

        .header h1 {{
            font-size: 24px;
            margin-bottom: 5px;
        }}

        .instrucciones-rapidas {{
            background: white;
            border-radius: 12px;
            padding: 15px 20px;
            margin-bottom: 20px;
            display: flex;
            justify-content: center;
            gap: 30px;
            flex-wrap: wrap;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        }}

        .instruccion-item {{
            display: flex;
            align-items: center;
            gap: 12px;
            font-size: 14px;
            font-weight: 500;
        }}

        .instruccion-color {{
            width: 24px;
            height: 24px;
            border-radius: 6px;
        }}

        .color-verde {{
            background: #d4edda;
            border: 2px solid #2e7d32;
        }}

        .color-rojo {{
            background: #ffebee;
            border: 2px solid #c62828;
        }}

        .instruccion-texto {{
            color: #333;
        }}

        .instruccion-texto strong {{
            color: {COLOR_FONDO};
        }}

        .estados-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fill, minmax(180px, 1fr));
            gap: 15px;
            max-height: 600px;
            overflow-y: auto;
            padding: 10px;
        }}

        .estado-btn {{
            background: {COLOR_FONDO};
            border: none;
            border-radius: 12px;
            padding: 15px 10px;
            color: white;
            font-weight: 600;
            cursor: pointer;
            transition: 0.3s;
            font-size: 14px;
        }}

        .estado-btn:hover {{
            background: {COLOR_SECUNDARIO};
            transform: translateY(-3px);
        }}

        .modal {{
            display: none;
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0,0,0,0.8);
            backdrop-filter: blur(5px);
            justify-content: center;
            align-items: center;
            z-index: 1000;
        }}

        .modal.active {{
            display: flex;
        }}
        
        .modal-form.active {{
            display: flex;
        }}

        .modal-content {{
            position: relative;
            background: linear-gradient(135deg, {COLOR_BORDE}dd);
            backdrop-filter: none;
            border-radius: 20px;
            padding: 30px;
            width: 90%;
            max-width: 450px;
            color: white;
            border: 1px solid rgba(255,255,255,0.2);
            animation: slideUp 0.3s;
        }}

        @keyframes slideUp {{
            from {{ transform: translateY(20px); opacity: 0; }}
            to {{ transform: translateY(0); opacity: 1; }}
        }}

        .modal h2 {{
            text-align: center;
            margin-bottom: 10px;
            font-size: 28px;
        }}

        .modal p {{
            text-align: center;
            margin-bottom: 20px;
            opacity: 0.9;
        }}

        .close-btn {{
            position: absolute;
            top: 10px;
            right: 15px;
            background: none;
            border: none;
            font-size: 28px;
            cursor: pointer;
            color: white;
            opacity: 0.8;
            transition: opacity 0.2s;
            line-height: 1;
        }}

        .close-btn:hover {{
            opacity: 1;
        }}

        .modal-form {{
            display: none;
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0,0,0,0.5);
            backdrop-filter: blur(5px);
            justify-content: center;
            align-items: center;
            z-index: 1000;
        }}

        .modal-form.active {{
            display: flex;
        }}

        .modal-form-content {{
            position: relative;
            background: white;
            border-radius: 20px;
            padding: 30px;
            width: 95%;
            max-width: 1400px;
            max-height: 90vh;
            overflow-x: auto;
            overflow-y: auto;
        }}

        .modal-form-content .close-btn {{
            color: #333;
            top: 10px;
            right: 15px;
        }}

        .form-group {{
            margin-bottom: 15px;
        }}

        .form-group label {{
            display: block;
            margin-bottom: 5px;
            font-weight: 600;
        }}

        .form-group input, .form-group select {{
            width: 100%;
            padding: 10px;
            background: rgba(255,255,255,0.2);
            border: 1px solid rgba(255,255,255,0.3);
            border-radius: 8px;
            color: white;
            font-size: 14px;
        }}

        .form-group input::placeholder {{
            color: rgba(255,255,255,0.6);
        }}

        .form-group input:focus {{
            outline: none;
            border-color: white;
            background: rgba(255,255,255,0.25);
        }}

        .help-text {{
            font-size: 12px;
            margin-top: 5px;
            opacity: 0.8;
        }}

        .btn {{
            width: 100%;
            padding: 12px;
            background: white;
            color: {COLOR_FONDO};
            border: none;
            border-radius: 8px;
            font-weight: 700;
            font-size: 16px;
            cursor: pointer;
            transition: 0.3s;
            margin-top: 10px;
        }}

        .btn:hover {{
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(0,0,0,0.3);
        }}

        .error {{
            color: #ffcccc;
            font-size: 13px;
            margin-top: 10px;
            text-align: center;
            display: none;
        }}

        .table-container {{
            overflow-x: auto;
            overflow-y: auto;
            max-height: 55vh;
            position: relative;
        }}

        .equipamiento-table {{
            width: 100%;
            border-collapse: collapse;
            position: relative;
        }}

        .equipamiento-table th, .equipamiento-table td {{
            border: 1px solid #ddd;
            padding: 10px 12px;
            text-align: center;
            vertical-align: middle;
        }}

        .equipamiento-table th:nth-child(1),
        .equipamiento-table td:nth-child(1) {{
            position: sticky;
            left: 0;
            background-color: {COLOR_FONDO};
            z-index: 10;
            min-width: 280px;
            max-width: 350px;
            text-align: left;
            color: white;
        }}

        .equipamiento-table td:nth-child(1) {{
            background-color: #f0f0f0;
            color: #333;
            font-weight: 500;
        }}

        .equipamiento-table th:nth-child(1) {{
            background-color: {COLOR_FONDO};
            color: white;
            z-index: 20;
        }}

        .equipamiento-table td:nth-child(1)::after,
        .equipamiento-table th:nth-child(1)::after {{
            content: '';
            position: absolute;
            top: 0;
            right: -5px;
            height: 100%;
            width: 5px;
            box-shadow: 2px 0 5px rgba(0,0,0,0.1);
            pointer-events: none;
        }}

        .equipamiento-table th {{
            background-color: {COLOR_FONDO};
            color: white;
            position: sticky;
            top: 0;
            z-index: 15;
            min-width: 120px;
        }}

        .equipamiento-table th:hover {{
            background-color: #1a8fbb !important;
            cursor: pointer;
        }}

        .equipamiento-table tr:nth-child(even) {{
            background-color: #f9f9f9;
        }}

        .equipamiento-table tbody tr:hover {{
            background-color: #e3f2fd !important;
        }}

        .equipamiento-table tbody tr:hover td:first-child {{
            background-color: #bbdef5 !important;
        }}

        .equipamiento-table td:hover {{
            background-color: #fff3e0 !important;
        }}

        .equipamiento-table td:hover .valor-mostrado {{
            transform: scale(1.05);
            box-shadow: 0 2px 8px rgba(0,0,0,0.15);
        }}

        .equipamiento-table tr.fila-guardada-bd {{
            background-color: #d4edda !important;
        }}

        .equipamiento-table tr.fila-guardada-bd:hover {{
            background-color: #c3e6cb !important;
        }}

        .equipamiento-table tr.fila-guardada-bd td:first-child {{
            background-color: #c3e6cb !important;
        }}

        .campo-container {{
            display: flex;
            gap: 8px;
            align-items: center;
            justify-content: center;
            flex-wrap: wrap;
        }}

        .valor-mostrado {{
            display: inline-block;
            background: #e8f5e9;
            color: #2e7d32;
            padding: 5px 10px;
            border-radius: 5px;
            font-weight: bold;
            font-size: 14px;
            min-width: 50px;
            text-align: center;
            cursor: pointer;
            transition: 0.2s;
        }}

        .valor-mostrado:hover {{
            background: #c8e6c9;
            transform: scale(1.02);
        }}

        .valor-vacio {{
            background: #ffebee;
            color: #c62828;
            cursor: pointer;
        }}

        .valor-vacio:hover {{
            background: #ffcdd2;
        }}

        .input-edicion {{
            width: 80px;
            padding: 5px;
            border: 2px solid {COLOR_SECUNDARIO};
            border-radius: 4px;
            text-align: center;
            font-size: 14px;
        }}

        .btn-accion {{
            color: white;
            border: none;
            padding: 5px 10px;
            border-radius: 5px;
            cursor: pointer;
            font-size: 12px;
            font-weight: 600;
            transition: 0.3s;
            white-space: nowrap;
            background: #2196F3;
        }}

        .btn-accion:hover {{
            background: #0b7dda;
        }}

        .btn-guardar {{
            background: {COLOR_SECUNDARIO};
            color: white;
            border: none;
            padding: 10px 20px;
            border-radius: 8px;
            cursor: pointer;
            font-size: 16px;
            font-weight: 600;
            transition: 0.3s;
        }}

        .btn-guardar:hover {{
            background: #0d6efd;
            transform: scale(1.02);
        }}

        .btn-guardar:disabled {{
            background: #999;
            cursor: not-allowed;
            transform: none;
        }}

        .btn-cancelar {{
            background: #666;
            color: white;
        }}

        .acciones {{
            display: flex;
            gap: 10px;
            margin-top: 20px;
            justify-content: center;
            flex-wrap: wrap;
        }}

        .progress {{
            margin-bottom: 20px;
            padding: 10px;
            background: #f0f0f0;
            border-radius: 8px;
            color: #333;
        }}

        .badge {{
            display: inline-block;
            padding: 3px 8px;
            border-radius: 12px;
            font-size: 11px;
            font-weight: bold;
        }}
        
        .badge-success {{
            background: #4CAF50;
            color: white;
        }}
        
        .badge-warning {{
            background: #ff9800;
            color: white;
        }}

        .save-indicator {{
            position: fixed;
            bottom: 20px;
            right: 20px;
            background: #4CAF50;
            color: white;
            padding: 10px 15px;
            border-radius: 8px;
            font-size: 14px;
            opacity: 0;
            transition: opacity 0.3s;
            z-index: 1000;
        }}

        .equipo-tooltip {{
            position: fixed;
            background: #0b5a7c;
            color: white;
            padding: 8px 15px;
            border-radius: 8px;
            font-size: 14px;
            font-weight: bold;
            z-index: 2000;
            pointer-events: none;
            box-shadow: 0 2px 10px rgba(0,0,0,0.2);
            white-space: nowrap;
            font-family: "League Spartan", sans-serif;
        }}

        .clues-selector {{
            margin-bottom: 20px;
            padding: 15px;
            background: #f8f9fa;
            border-radius: 8px;
            display: flex;
            align-items: center;
            gap: 15px;
            flex-wrap: wrap;
        }}

        .clues-selector label {{
            font-weight: 600;
            margin-right: 10px;
        }}

        .clues-selector select {{
            padding: 8px 15px;
            border-radius: 5px;
            border: 1px solid #ddd;
            font-size: 14px;
            min-width: 300px;
            background: white;
            flex: 1;
        }}

        .clues-selector button {{
            padding: 8px 20px;
            background: {COLOR_SECUNDARIO};
            color: white;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            font-weight: 600;
            transition: 0.3s;
        }}

        .clues-selector button:hover {{
            background: #0d6efd;
            transform: scale(1.02);
        }}

        .consultorios-config {{
            background: #fff8e1;
            padding: 15px 20px;
            border-radius: 8px;
            margin-bottom: 20px;
            border-left: 4px solid #ff9800;
            display: flex;
            align-items: center;
            gap: 20px;
            flex-wrap: wrap;
        }}

        .consultorios-config label {{
            font-weight: 600;
            color: #333;
        }}

        .consultorios-config input {{
            width: 80px;
            padding: 8px 12px;
            border: 2px solid #ff9800;
            border-radius: 5px;
            font-size: 16px;
            text-align: center;
        }}

        .consultorios-config button {{
            padding: 8px 20px;
            background: #ff9800;
            color: white;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            font-weight: 600;
            transition: 0.3s;
        }}

        .consultorios-config button:hover {{
            background: #e68900;
            transform: scale(1.02);
        }}

        .consultorios-config .info-text {{
            color: #666;
            font-size: 13px;
        }}

        .unidad-info {{
            background: #e3f2fd;
            padding: 12px 18px;
            border-radius: 8px;
            margin-bottom: 15px;
            border-left: 4px solid {COLOR_FONDO};
            display: flex;
            flex-wrap: wrap;
            gap: 20px;
            align-items: center;
        }}

        .unidad-info-item {{
            display: flex;
            align-items: center;
            gap: 8px;
        }}

        .unidad-info-item strong {{
            color: {COLOR_FONDO};
        }}

        .equipo-nombre {{
            font-size: 13px;
            line-height: 1.3;
        }}

        .popup-unidad {{
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0,0,0,0.6);
            backdrop-filter: blur(3px);
            display: none;
            justify-content: center;
            align-items: center;
            z-index: 3000;
        }}

        .popup-unidad.active {{
            display: flex;
        }}

        .popup-content {{
            background: white;
            border-radius: 15px;
            padding: 30px;
            max-width: 500px;
            width: 90%;
            box-shadow: 0 10px 40px rgba(0,0,0,0.3);
            animation: slideUp 0.3s;
        }}

        .popup-content h3 {{
            color: {COLOR_FONDO};
            margin-bottom: 15px;
            text-align: center;
        }}

        .popup-content p {{
            margin: 8px 0;
            color: #333;
        }}

        .popup-content .popup-close {{
            margin-top: 20px;
            padding: 10px 30px;
            background: {COLOR_FONDO};
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            font-weight: 600;
            width: 100%;
            transition: 0.3s;
        }}

        .popup-content .popup-close:hover {{
            background: {COLOR_SECUNDARIO};
        }}
    </style>
</head>
<body>
    <div class="menu-container">
        <button class="menu-btn" onclick="toggleMenu()">
            <span></span>
            <span></span>
            <span></span>
        </button>
    </div>

    <div class="menu-overlay" id="menuOverlay" onclick="toggleMenu()"></div>
    
    <div class="menu-panel" id="menuPanel">
        <button class="close-menu" onclick="toggleMenu()">&times;</button>
        <h2>Instrucciones</h2>
        
        <h3>1. Seleccionar Estado</h3>
        <p>Haga clic en el boton del estado correspondiente para comenzar el registro de equipamiento.</p>
        
        <h3>2. Registrar Datos del Usuario</h3>
        <p>Complete el formulario con nombre y correo electronico.</p>
        
        <h3>3. Seleccionar Unidad Medica</h3>
        <p>Elija la unidad medica especifica que desea registrar en el selector desplegable.</p>
        
        <h3>4. Configurar Consultorios</h3>
        <p>Ingrese el numero de consultorios que tiene la unidad y presione "Aplicar".</p>
        
        <h3>5. Llenar Equipamiento</h3>
        <p>Haga clic en cualquier celda para editarla, ingrese el valor y presione ACEPTAR.</p>
        <p><strong style="color:#2e7d32">VERDE:</strong> Valor ya registrado - Haga clic para modificar</p>
        <p><strong style="color:#c62828">ROJO:</strong> Campo pendiente - Haga clic para llenar</p>
        <p><strong style="color:#d4edda">FILA VERDE:</strong> Unidad ya guardada en la base de datos</p>
        
        <h3>6. Guardar Unidad</h3>
        <p>Una vez que haya llenado los campos, presione el boton <strong>"Guardar unidad"</strong>.</p>
        
        <h3>7. Ver Detalles</h3>
        <p>Haga clic en <strong>"Ver detalles"</strong> para mostrar informacion de la unidad.</p>
        
        <h3>Consejos</h3>
        <ul>
            <li>Use solo numeros enteros</li>
            <li>Debe presionar ACEPTAR en cada celda antes de guardar</li>
            <li>El progreso local se guarda automaticamente</li>
            <li>Pase el mouse sobre cualquier celda para ver que equipo esta llenando</li>
            <li>Las columnas de consultorios se generan dinamicamente</li>
        </ul>
        
        <h3>Soporte</h3>
        <p>Si tiene problemas, contacte al area de sistemas de IMSS Bienestar.</p>
    </div>

    <div class="container">
        <div class="header">
            <img src="https://imssbienestar.gob.mx/assets/img/imb_b.svg" alt="IMSS Bienestar">
            <h1>CUESTIONARIO DE EQUIPAMIENTO POR UNIDAD MEDICA</h1>
            <p>Seleccione una entidad para registrar el equipamiento de sus unidades</p>
        </div>

        <div class="estados-grid" id="estadosGrid">
'''

# Generar botones de entidades
for entidad in entidades_unicas:
    num_unidades = len(base[base['entidad'] == entidad])
    html_content += f'''
            <button class="estado-btn" onclick="abrirModal('{entidad}')">
                {entidad}<br>
                <small style="font-size: 11px;">{num_unidades} unidades</small>
            </button>
    '''

html_content += f'''
        </div>
    </div>

    <div class="modal" id="loginModal">
        <div class="modal-content">
            <button class="close-btn" onclick="cerrarModal()">&times;</button>
            <h2 id="modalEstado"></h2>
            <p>Registre sus datos para continuar</p>
            
            <div class="form-group">
                <label>Entidad</label>
                <input type="text" id="usuarioInput" readonly>
            </div>
            
            <div class="form-group">
                <label>Nombre completo</label>
                <input type="text" id="nombreInput" placeholder="Ej: Juan Carlos Perez Gonzalez" required>
                <div class="help-text">Ingrese nombre(s) y apellidos completos</div>
            </div>
            
            <div class="form-group">
                <label>Correo electronico institucional</label>
                <input type="email" id="emailInput" placeholder="ejemplo@imssbienestar.gob.mx" required>
                <div class="help-text">Ingrese su correo electronico</div>
            </div>
            
            <button class="btn" onclick="validarDatos()">Continuar</button>
            <div id="errorMsg" class="error"></div>
        </div>
    </div>

    <div class="popup-unidad" id="popupUnidad">
        <div class="popup-content">
            <h3>Detalles de la Unidad</h3>
            <div id="popupInfo">
                <p><strong>CLUES:</strong> <span id="popupClues"></span></p>
                <p><strong>Unidad Medica:</strong> <span id="popupNombre"></span></p>
                <p><strong>Categoria:</strong> <span id="popupCategoria"></span></p>
                <p><strong>Entidad:</strong> <span id="popupEntidad"></span></p>
                <p><strong>Registrado por:</strong> <span id="popupUsuario"></span></p>
                <p><strong>Consultorios:</strong> <span id="popupConsultorios"></span></p>
            </div>
            <button class="popup-close" onclick="cerrarPopup()">Cerrar</button>
        </div>
    </div>

    <div class="modal-form" id="formModal">
        <div class="modal-form-content">
            <button class="close-btn" onclick="cerrarFormModal()">&times;</button>
            <h2 id="formEstado"></h2>
            <div id="userInfo" style="background: #f0f0f0; padding: 10px; border-radius: 8px; margin-bottom: 15px; font-size: 14px;"></div>
            
            <div class="clues-selector">
                <label for="cluesSelector">Seleccionar Unidad Medica:</label>
                <select id="cluesSelector" onchange="cambiarUnidadSeleccionada()">
                    <option value="">-- Seleccione una unidad --</option>
                </select>
                <button onclick="cargarUnidades(estadoSeleccionado)">Recargar</button>
                <button onclick="mostrarPopupUnidad()" style="background:#17a2b8;">Ver detalles</button>
            </div>
            
            <div id="unidadInfo" class="unidad-info" style="display:none;">
                <span class="unidad-info-item"><strong>CLUES:</strong> <span id="cluesUnidadSeleccionada"></span></span>
                <span class="unidad-info-item"><strong>Unidad:</strong> <span id="nombreUnidadSeleccionada"></span></span>
                <span class="unidad-info-item"><strong>Categoria:</strong> <span id="categoriaUnidadSeleccionada"></span></span>
            </div>
            
            <div class="consultorios-config">
                <label for="numConsultorios">Numero de consultorios:</label>
                <input type="number" id="numConsultorios" min="0" max="20" value="1">
                <button onclick="aplicarConsultorios()">Aplicar</button>
                <span class="info-text" id="consultoriosInfo">Consultorios configurados: 1</span>
            </div>
            
            <div id="progressInfo" class="progress"></div>
            
            <div class="instrucciones-rapidas">
                <div class="instruccion-item">
                    <div class="instruccion-color color-verde"></div>
                    <div class="instruccion-texto"><strong>VERDE</strong> Haga clic para MODIFICAR</div>
                </div>
                <div class="instruccion-item">
                    <div class="instruccion-color color-rojo"></div>
                    <div class="instruccion-texto"><strong>ROJO</strong> Haga clic para LLENAR</div>
                </div>
                <div class="instruccion-item">
                    <div style="background: #d4edda; width: 24px; height: 24px; border-radius: 6px; border: 1px solid #2e7d32;"></div>
                    <div class="instruccion-texto"><strong>FILA VERDE</strong> Unidad ya guardada</div>
                </div>
                <div class="instruccion-item">
                    <div style="background: #2196F3; width: 24px; height: 24px; border-radius: 6px;"></div>
                    <div class="instruccion-texto"><strong>ACEPTAR</strong> Para confirmar el valor</div>
                </div>
                <div class="instruccion-item">
                    <div style="background: {COLOR_SECUNDARIO}; width: 24px; height: 24px; border-radius: 6px;"></div>
                    <div class="instruccion-texto"><strong>GUARDAR</strong> Envia los datos a la nube</div>
                </div>
            </div>
            
            <div class="table-container">
                <table class="equipamiento-table" id="equipamientoTable">
                    <thead id="tableHead">
                        <tr>
                            <th style="min-width: 280px; max-width: 350px;">Equipo</th>
                        </tr>
                    </thead>
                    <tbody id="tableBody">
                    </tbody>
                </table>
            </div>
            
            <div style="margin-top: 20px; text-align: center;">
                <button class="btn-guardar" id="btnGuardarUnidad" onclick="guardarUnidadEnNube()">Guardar unidad</button>
            </div>
            
            <div class="acciones">
                <button type="button" class="btn btn-cancelar" onclick="cerrarFormModal()">Cerrar</button>
            </div>
        </div>
    </div>

    <div class="save-indicator" id="saveIndicator">
        Unidad guardada
    </div>

    <script>
        let celdaEnEdicion = null;
        let tooltip = null;
        let filasGuardadasBD = new Set();
        let unidadSeleccionada = null;
        let numConsultorios = 1;

        function crearTooltip() {{
            if (!tooltip) {{
                tooltip = document.createElement('div');
                tooltip.className = 'equipo-tooltip';
                tooltip.style.display = 'none';
                document.body.appendChild(tooltip);
            }}
            return tooltip;
        }}

        function mostrarTooltip(event, texto) {{
            const tooltip = crearTooltip();
            tooltip.textContent = texto;
            tooltip.style.display = 'block';
            let left = event.pageX + 15;
            let top = event.pageY - 30;
            
            if (left + tooltip.offsetWidth > window.innerWidth) {{
                left = event.pageX - tooltip.offsetWidth - 15;
            }}
            if (top < 0) {{
                top = event.pageY + 20;
            }}
            
            tooltip.style.left = left + 'px';
            tooltip.style.top = top + 'px';
        }}

        function ocultarTooltip() {{
            if (tooltip) {{
                tooltip.style.display = 'none';
            }}
        }}

        function toggleMenu() {{
            const panel = document.getElementById('menuPanel');
            const overlay = document.getElementById('menuOverlay');
            panel.classList.toggle('active');
            overlay.classList.toggle('active');
        }}

        document.addEventListener('keydown', function(e) {{
            if (e.key === 'Escape') {{
                const panel = document.getElementById('menuPanel');
                const overlay = document.getElementById('menuOverlay');
                panel.classList.remove('active');
                overlay.classList.remove('active');
                if (celdaEnEdicion === null) {{
                    cerrarModal();
                    cerrarFormModal();
                    cerrarPopup();
                }}
                ocultarTooltip();
            }}
        }});

        const datosUnidades = {json.dumps({entidad: get_unidades_por_entidad(entidad) for entidad in entidades_unicas}, ensure_ascii=False)};
        const columnasEquipamiento = {json.dumps(columnas_equipamiento)};
        
        let estadoSeleccionado = '';
        let datosActuales = [];
        let usuarioActual = {{
            nombre: '',
            email: '',
            entidad: ''
        }};

        function formatearNombreEquipo(nombre) {{
            if (!nombre) return '';
            let formateado = nombre.replace(/_/g, ' ');
            formateado = formateado.split(' ').map(palabra => 
                palabra.charAt(0).toUpperCase() + palabra.slice(1).toLowerCase()
            ).join(' ');
            return formateado;
        }}

        function formatearNombre(nombre) {{
            if (!nombre) return '';
            let formateado = nombre.replace(/_/g, ' ');
            formateado = formateado.split(' ').map(palabra => 
                palabra.charAt(0).toUpperCase() + palabra.slice(1).toLowerCase()
            ).join(' ');
            return formateado;
        }}

        function guardarProgresoLocal() {{
            if (estadoSeleccionado && datosActuales.length > 0 && usuarioActual.email) {{
                const clave = `equipamiento_${{estadoSeleccionado}}_${{usuarioActual.email}}`;
                const progreso = {{
                    entidad: estadoSeleccionado,
                    usuario: usuarioActual,
                    datos: datosActuales,
                    numConsultorios: numConsultorios,
                    fecha_guardado: new Date().toISOString()
                }};
                localStorage.setItem(clave, JSON.stringify(progreso));
            }}
        }}

        function cargarProgresoLocal(estado, email) {{
            const clave = `equipamiento_${{estado}}_${{email}}`;
            const guardado = localStorage.getItem(clave);
            if (guardado) {{
                try {{
                    const progreso = JSON.parse(guardado);
                    return progreso;
                }} catch(e) {{
                    return null;
                }}
            }}
            return null;
        }}

        function verificarProgresoGuardado(estado, email) {{
            const clave = `equipamiento_${{estado}}_${{email}}`;
            const guardado = localStorage.getItem(clave);
            if (guardado) {{
                try {{
                    const progreso = JSON.parse(guardado);
                    const fecha = new Date(progreso.fecha_guardado);
                    return {{
                        existe: true,
                        fecha: fecha.toLocaleString()
                    }};
                }} catch(e) {{
                    return {{ existe: false }};
                }}
            }}
            return {{ existe: false }};
        }}

        function abrirModal(estado) {{
            estadoSeleccionado = estado;
            document.getElementById('modalEstado').textContent = estado;
            document.getElementById('usuarioInput').value = estado;
            document.getElementById('nombreInput').value = '';
            document.getElementById('emailInput').value = '';
            document.getElementById('errorMsg').style.display = 'none';
            document.getElementById('loginModal').classList.add('active');
            document.getElementById('nombreInput').focus();
        }}

        function cerrarModal() {{
            document.getElementById('loginModal').classList.remove('active');
        }}

        function validarNombreCompleto(nombre) {{
            const nombreTrim = nombre.trim();
            const partes = nombreTrim.split(/\\s+/);
            if (partes.length < 2) return false;
            if (partes.length > 5) return false;
            for (let parte of partes) {{
                if (parte.length < 2) return false;
            }}
            return true;
        }}

        function validarDatos() {{
            const nombre = document.getElementById('nombreInput').value;
            const email = document.getElementById('emailInput').value.trim();
            
            if (!validarNombreCompleto(nombre)) {{
                document.getElementById('errorMsg').textContent = 'Por favor, ingrese su nombre completo';
                document.getElementById('errorMsg').style.display = 'block';
                return;
            }}
            if (email === '') {{
                document.getElementById('errorMsg').textContent = 'Por favor, ingrese un correo electronico';
                document.getElementById('errorMsg').style.display = 'block';
                return;
            }}
            
            const emailRegex = /^[^\\s@]+@([^\\s@]+\\.)+[^\\s@]+$/;
            if (!emailRegex.test(email)) {{
                document.getElementById('errorMsg').textContent = 'Por favor, ingrese un correo electronico valido';
                document.getElementById('errorMsg').style.display = 'block';
                return;
            }}
            
            usuarioActual = {{
                nombre: nombre.trim(),
                email: email,
                entidad: estadoSeleccionado
            }};
            
            document.getElementById('errorMsg').style.display = 'none';
            cerrarModal();
            abrirFormulario(estadoSeleccionado);
        }}

        function abrirFormulario(estado) {{
            document.getElementById('formEstado').innerHTML = `Cuestionario de Equipamiento - <strong>${{estado}}</strong>`;
            document.getElementById('userInfo').innerHTML = `
                <strong>Registrado por:</strong> ${{usuarioActual.nombre}} | 
                <strong>Correo:</strong> ${{usuarioActual.email}} | 
                <strong>Entidad:</strong> ${{usuarioActual.entidad}}
            `;
            document.getElementById('formModal').classList.add('active');
            
            const progreso = verificarProgresoGuardado(estado, usuarioActual.email);
            
            if (progreso.existe) {{
                const restaurar = confirm(`Se encontro un progreso guardado del ${{progreso.fecha}}.\\n\\n¿Desea continuar donde lo dejo?`);
                if (restaurar) {{
                    cargarUnidadesConProgreso(estado);
                }} else {{
                    cargarUnidades(estado);
                }}
            }} else {{
                cargarUnidades(estado);
            }}
        }}

        function cargarUnidadesConProgreso(estado) {{
            const unidades = datosUnidades[estado] || [];
            const progresoCompleto = cargarProgresoLocal(estado, usuarioActual.email);
            
            if (progresoCompleto && progresoCompleto.datos && progresoCompleto.datos.length === unidades.length) {{
                datosActuales = JSON.parse(JSON.stringify(progresoCompleto.datos));
                numConsultorios = progresoCompleto.numConsultorios || 1;
                document.getElementById('numConsultorios').value = numConsultorios;
                document.getElementById('consultoriosInfo').textContent = 'Consultorios configurados: ' + numConsultorios;
            }} else {{
                datosActuales = JSON.parse(JSON.stringify(unidades));
            }}
            
            actualizarSelectorUnidades();
            if (unidadSeleccionada) {{
                mostrarUnidadSeleccionada();
            }}
            
            const progressInfo = document.getElementById('progressInfo');
            const mensajeRestaurado = document.createElement('div');
            mensajeRestaurado.style.background = '#e8f5e9';
            mensajeRestaurado.style.color = '#2e7d32';
            mensajeRestaurado.style.padding = '8px';
            mensajeRestaurado.style.borderRadius = '8px';
            mensajeRestaurado.style.marginTop = '10px';
            mensajeRestaurado.style.textAlign = 'center';
            mensajeRestaurado.innerHTML = 'Progreso anterior restaurado correctamente';
            progressInfo.appendChild(mensajeRestaurado);
            setTimeout(() => {{
                mensajeRestaurado.remove();
            }}, 3000);
        }}

        function actualizarSelectorUnidades() {{
            const selector = document.getElementById('cluesSelector');
            const valorActual = selector.value;
            selector.innerHTML = '<option value="">-- Seleccione una unidad --</option>';
            
            for (let i = 0; i < datosActuales.length; i++) {{
                const unidad = datosActuales[i];
                const option = document.createElement('option');
                option.value = unidad.clues_imb || i;
                const label = unidad.clues_imb + ' - ' + (unidad.nombre_de_la_unidad || 'Sin nombre');
                option.textContent = label.length > 60 ? label.substring(0, 57) + '...' : label;
                selector.appendChild(option);
            }}
            
            if (valorActual && document.querySelector(`#cluesSelector option[value="${{valorActual}}"]`)) {{
                selector.value = valorActual;
            }}
        }}

        function cambiarUnidadSeleccionada() {{
            const selector = document.getElementById('cluesSelector');
            const valorSeleccionado = selector.value;
            
            if (!valorSeleccionado) {{
                unidadSeleccionada = null;
                document.getElementById('unidadInfo').style.display = 'none';
                document.getElementById('tableBody').innerHTML = '';
                document.getElementById('progressInfo').innerHTML = '<p>Seleccione una unidad medica para comenzar</p>';
                document.getElementById('btnGuardarUnidad').disabled = true;
                return;
            }}
            
            unidadSeleccionada = datosActuales.find(u => u.clues_imb === valorSeleccionado) || 
                               datosActuales[parseInt(valorSeleccionado)];
            
            if (unidadSeleccionada) {{
                mostrarUnidadSeleccionada();
                document.getElementById('btnGuardarUnidad').disabled = false;
            }}
        }}

        function mostrarUnidadSeleccionada() {{
            if (!unidadSeleccionada) return;
            
            document.getElementById('nombreUnidadSeleccionada').textContent = unidadSeleccionada.nombre_de_la_unidad || 'Sin nombre';
            document.getElementById('cluesUnidadSeleccionada').textContent = unidadSeleccionada.clues_imb || 'Sin CLUES';
            document.getElementById('categoriaUnidadSeleccionada').textContent = unidadSeleccionada.categoria_gerencial_ampliada || 'Sin categoria';
            document.getElementById('unidadInfo').style.display = 'flex';
            
            aplicarConsultorios();
        }}

        function aplicarConsultorios() {{
            const input = document.getElementById('numConsultorios');
            let valor = parseInt(input.value);
            
            if (isNaN(valor) || valor < 0) {{
                valor = 0;
                input.value = 0;
            }}
            if (valor > 20) {{
                valor = 20;
                input.value = 20;
                alert('El maximo de consultorios permitido es 20');
            }}
            
            numConsultorios = valor;
            document.getElementById('consultoriosInfo').textContent = 'Consultorios configurados: ' + numConsultorios;
            
            if (unidadSeleccionada) {{
                mostrarUnidadTabla(unidadSeleccionada);
                actualizarProgresoUnidad();
                guardarProgresoLocal();
            }}
        }}

        function mostrarUnidadTabla(unidad) {{
            const tbody = document.getElementById('tableBody');
            const thead = document.getElementById('tableHead');
            tbody.innerHTML = '';
            
            const nombreUnidad = unidad.nombre_de_la_unidad || 'Sin nombre';
            const cluesUnidad = unidad.clues_imb || 'Sin CLUES';
            
            const claveUnidad = usuarioActual.email + '|' + unidad.clues_imb;
            const estaGuardada = filasGuardadasBD.has(claveUnidad);
            
            // Construir encabezados - SOLO COLUMNAS DE CONSULTORIOS (sin color especial)
            let headerRow = '<tr><th style="min-width: 280px; max-width: 350px;">Equipo</th>';
            
            // Columnas por consultorio - MISMO COLOR que el resto del formulario
            for (let i = 1; i <= numConsultorios; i++) {{
                headerRow += '<th style="min-width: 100px;">Consultorio ' + i + '</th>';
            }}
            
            headerRow += '</tr>';
            thead.innerHTML = headerRow;
            
            // Generar filas - FILTRANDO Consultorios Generales
            const columnasFiltradas = columnasEquipamiento.filter(col => 
                col !== 'Consultorios_Generales' && col !== 'Consultorios_Generales_Habilitados'
            );
            
            for (let i = 0; i < columnasFiltradas.length; i++) {{
                const col = columnasFiltradas[i];
                const row = tbody.insertRow();
                
                if (estaGuardada) {{
                    row.classList.add('fila-guardada-bd');
                }}
                
                const nombreEquipo = formatearNombreEquipo(col);
                
                // Columna de equipo (fija)
                const cellEquipo = row.insertCell(0);
                cellEquipo.innerHTML = `<span class="equipo-nombre">${{nombreEquipo}}</span>`;
                cellEquipo.style.backgroundColor = estaGuardada ? '#c3e6cb' : '#f0f0f0';
                
                cellEquipo.addEventListener('mouseenter', function(e) {{
                    mostrarTooltip(e, 'Equipo: ' + nombreEquipo);
                }});
                cellEquipo.addEventListener('mousemove', function(e) {{
                    mostrarTooltip(e, 'Equipo: ' + nombreEquipo);
                }});
                cellEquipo.addEventListener('mouseleave', function() {{
                    ocultarTooltip();
                }});
                
                // Columnas por consultorio
                for (let j = 1; j <= numConsultorios; j++) {{
                    const cellConsultorio = row.insertCell();
                    const colConsultorio = col + '_consultorio_' + j;
                    const valorConsultorio = unidad[colConsultorio] || null;
                    const filaOriginal = datosActuales.indexOf(unidad);
                    const campoConsultorio = crearCampoEquipamiento(valorConsultorio, filaOriginal, colConsultorio, 'consultorio_' + j);
                    cellConsultorio.appendChild(campoConsultorio);
                    
                    cellConsultorio.addEventListener('mouseenter', function(e) {{
                        this.style.backgroundColor = '#fff3e0';
                        mostrarTooltip(e, nombreEquipo + ' (Consultorio ' + j + ')\\nUnidad: ' + nombreUnidad);
                    }});
                    cellConsultorio.addEventListener('mousemove', function(e) {{
                        mostrarTooltip(e, nombreEquipo + ' (Consultorio ' + j + ')\\nUnidad: ' + nombreUnidad);
                    }});
                    cellConsultorio.addEventListener('mouseleave', function() {{
                        this.style.backgroundColor = '';
                        ocultarTooltip();
                    }});
                }}
            }}
        }}

        function crearCampoEquipamiento(valorActual, filaOriginal, columna, tipo) {{
            const container = document.createElement('div');
            container.className = 'campo-container';
            
            const valorMostrado = document.createElement('div');
            valorMostrado.className = 'valor-mostrado';
            
            const tieneValor = (valorActual !== null && valorActual !== undefined && !isNaN(valorActual) && valorActual !== 0 && valorActual !== '');
            const valorDisplay = tieneValor ? valorActual : 'PENDIENTE';
            
            valorMostrado.textContent = valorDisplay;
            
            if (!tieneValor) {{
                valorMostrado.classList.add('valor-vacio');
            }}
            
            const iniciarEdicion = () => {{
                if (celdaEnEdicion !== null) {{
                    alert('Primero presione ACEPTAR en la celda que esta editando');
                    return;
                }}
                
                celdaEnEdicion = container;
                container.innerHTML = '';
                
                const inputField = document.createElement('input');
                inputField.type = 'number';
                inputField.step = '1';
                inputField.value = (tieneValor && valorActual !== null) ? valorActual : '';
                inputField.placeholder = '0';
                inputField.className = 'input-edicion';
                
                const btnAceptar = document.createElement('button');
                btnAceptar.className = 'btn-accion';
                btnAceptar.textContent = 'ACEPTAR';
                
                const finalizarEdicion = () => {{
                    const nuevoValor = inputField.value;
                    let numero = parseInt(nuevoValor);
                    
                    if (nuevoValor === '') {{
                        datosActuales[filaOriginal][columna] = null;
                    }} else if (!isNaN(numero) && numero >= 0) {{
                        datosActuales[filaOriginal][columna] = numero;
                    }} else {{
                        alert('Ingrese un numero valido');
                        return;
                    }}
                    
                    celdaEnEdicion = null;
                    guardarProgresoLocal();
                    if (unidadSeleccionada) {{
                        mostrarUnidadTabla(unidadSeleccionada);
                    }}
                    actualizarProgresoUnidad();
                }};
                
                btnAceptar.onclick = finalizarEdicion;
                inputField.onkeypress = (e) => {{
                    if (e.key === 'Enter') {{
                        finalizarEdicion();
                    }}
                }};
                
                container.appendChild(inputField);
                container.appendChild(btnAceptar);
                inputField.focus();
            }};
            
            valorMostrado.onclick = iniciarEdicion;
            container.appendChild(valorMostrado);
            
            return container;
        }}

        function guardarUnidadEnNube() {{
            if (!unidadSeleccionada) {{
                alert('Seleccione una unidad medica primero');
                return;
            }}
            
            const btn = document.getElementById('btnGuardarUnidad');
            btn.disabled = true;
            btn.textContent = 'Guardando...';
            
            const datosParaGuardar = {{
                entidad: estadoSeleccionado,
                usuario_nombre: usuarioActual.nombre,
                usuario_email: usuarioActual.email,
                clues_imb: unidadSeleccionada.clues_imb,
                nombre_de_la_unidad: unidadSeleccionada.nombre_de_la_unidad,
                categoria: unidadSeleccionada.categoria_gerencial_ampliada,
                num_consultorios: numConsultorios
            }};
            
            // Guardar cantidades por consultorio
            const columnasFiltradas = columnasEquipamiento.filter(col => 
                col !== 'Consultorios_Generales' && col !== 'Consultorios_Generales_Habilitados'
            );
            
            for (let i = 0; i < columnasFiltradas.length; i++) {{
                const col = columnasFiltradas[i];
                for (let j = 1; j <= numConsultorios; j++) {{
                    const colConsultorio = col + '_consultorio_' + j;
                    const valor = unidadSeleccionada[colConsultorio];
                    datosParaGuardar[colConsultorio] = (valor !== null && valor !== undefined && !isNaN(valor) && valor !== 0 && valor !== '') ? valor : '';
                }}
            }}
            
            const scriptURL = '{script_url}';
            
            fetch(scriptURL, {{ 
                method: 'POST', 
                mode: 'no-cors',
                headers: {{ 'Content-Type': 'application/json' }},
                body: JSON.stringify(datosParaGuardar)
            }})
            .then(() => {{
                const claveUnidad = usuarioActual.email + '|' + unidadSeleccionada.clues_imb;
                filasGuardadasBD.add(claveUnidad);
                
                btn.disabled = false;
                btn.textContent = 'Guardado';
                btn.style.background = '#28a745';
                
                const indicator = document.getElementById('saveIndicator');
                indicator.style.background = '#28a745';
                indicator.textContent = 'Unidad guardada en la nube';
                indicator.style.opacity = '1';
                
                const rows = document.querySelectorAll('#equipamientoTable tbody tr');
                rows.forEach(row => {{
                    row.classList.add('fila-guardada-bd');
                    const firstCell = row.cells[0];
                    if (firstCell) {{
                        firstCell.style.backgroundColor = '#c3e6cb';
                    }}
                }});
                
                setTimeout(() => {{
                    indicator.style.opacity = '0';
                    setTimeout(() => {{
                        btn.textContent = 'Guardar unidad';
                        btn.style.background = '{COLOR_SECUNDARIO}';
                    }}, 2000);
                }}, 2000);
            }})
            .catch(error => {{
                console.error('Error:', error);
                btn.disabled = false;
                btn.textContent = 'Guardar unidad';
                alert('Error al guardar. Revisa tu conexion.');
            }});
        }}

        function cargarUnidades(estado) {{
            const unidades = datosUnidades[estado] || [];
            datosActuales = JSON.parse(JSON.stringify(unidades));
            filasGuardadasBD.clear();
            
            actualizarSelectorUnidades();
            
            if (datosActuales.length > 0) {{
                const selector = document.getElementById('cluesSelector');
                selector.value = datosActuales[0].clues_imb || '0';
                cambiarUnidadSeleccionada();
            }}
        }}
        
        function actualizarProgresoUnidad() {{
            if (!unidadSeleccionada) {{
                document.getElementById('progressInfo').innerHTML = '<p>Seleccione una unidad medica para ver el progreso</p>';
                return;
            }}
            
            const columnasFiltradas = columnasEquipamiento.filter(col => 
                col !== 'Consultorios_Generales' && col !== 'Consultorios_Generales_Habilitados'
            );
            
            let completados = 0;
            let totalCampos = columnasFiltradas.length * numConsultorios;
            
            // Contar por consultorio
            for (let c = 0; c < columnasFiltradas.length; c++) {{
                const col = columnasFiltradas[c];
                for (let j = 1; j <= numConsultorios; j++) {{
                    const colConsultorio = col + '_consultorio_' + j;
                    const valor = unidadSeleccionada[colConsultorio];
                    if (valor !== null && valor !== undefined && !isNaN(valor) && valor !== 0 && valor !== '') {{
                        completados++;
                    }}
                }}
            }}
            
            const porcentaje = totalCampos > 0 ? Math.round((completados / totalCampos) * 100) : 0;
            let badgeClass = porcentaje === 100 ? 'badge-success' : 'badge-warning';
            let badgeText = porcentaje === 100 ? 'COMPLETO' : 'PENDIENTE';
            
            document.getElementById('progressInfo').innerHTML = `
                <strong>Progreso de llenado</strong>
                <div style="background: #ddd; border-radius: 10px; margin-top: 5px;">
                    <div style="background: {COLOR_SECUNDARIO}; width: ` + porcentaje + `%; height: 20px; border-radius: 10px; transition: width 0.3s;"></div>
                </div>
                <p style="margin-top: 5px;">` + completados + ` de ` + totalCampos + ` campos registrados (` + porcentaje + `%)</p>
                <span class="badge ` + badgeClass + `">
                    ` + badgeText + `
                </span>
                <p style="font-size: 13px; color: #666; margin-top: 5px;">
                    ` + columnasFiltradas.length + ` equipos x ` + numConsultorios + ` consultorios
                </p>
            `;
        }}

        function mostrarPopupUnidad() {{
            if (!unidadSeleccionada) {{
                alert('Seleccione una unidad medica primero');
                return;
            }}
            document.getElementById('popupClues').textContent = unidadSeleccionada.clues_imb || 'Sin CLUES';
            document.getElementById('popupNombre').textContent = unidadSeleccionada.nombre_de_la_unidad || 'Sin nombre';
            document.getElementById('popupCategoria').textContent = unidadSeleccionada.categoria_gerencial_ampliada || 'Sin categoria';
            document.getElementById('popupEntidad').textContent = estadoSeleccionado;
            document.getElementById('popupUsuario').textContent = usuarioActual.nombre + ' (' + usuarioActual.email + ')';
            document.getElementById('popupConsultorios').textContent = numConsultorios;
            document.getElementById('popupUnidad').classList.add('active');
        }}

        function cerrarPopup() {{
            document.getElementById('popupUnidad').classList.remove('active');
        }}

        function cerrarFormModal() {{
            if (celdaEnEdicion !== null) {{
                alert('Primero presione ACEPTAR en la celda que esta editando');
                return;
            }}
            document.getElementById('formModal').classList.remove('active');
            ocultarTooltip();
        }}

        document.getElementById('nombreInput').addEventListener('input', function() {{
            document.getElementById('errorMsg').style.display = 'none';
        }});
        
        document.getElementById('emailInput').addEventListener('input', function() {{
            document.getElementById('errorMsg').style.display = 'none';
        }});

        document.getElementById('loginModal').addEventListener('click', function(e) {{
            if (e.target === this) {{
                cerrarModal();
            }}
        }});

        document.getElementById('emailInput').addEventListener('keypress', function(e) {{
            if (e.key === 'Enter') {{
                validarDatos();
            }}
        }});
        
        document.getElementById('nombreInput').addEventListener('keypress', function(e) {{
            if (e.key === 'Enter') {{
                document.getElementById('emailInput').focus();
            }}
        }});
        
        document.addEventListener('mousemove', function(e) {{
            if (tooltip && tooltip.style.display === 'block') {{
                let left = e.pageX + 15;
                let top = e.pageY - 30;
                if (left + tooltip.offsetWidth > window.innerWidth) {{
                    left = e.pageX - tooltip.offsetWidth - 15;
                }}
                if (top < 0) {{
                    top = e.pageY + 20;
                }}
                tooltip.style.left = left + 'px';
                tooltip.style.top = top + 'px';
            }}
        }});

        document.getElementById('popupUnidad').addEventListener('click', function(e) {{
            if (e.target === this) {{
                cerrarPopup();
            }}
        }});

        // Enter en el input de consultorios
        document.getElementById('numConsultorios').addEventListener('keypress', function(e) {{
            if (e.key === 'Enter') {{
                aplicarConsultorios();
            }}
        }});
    </script>
</body>
</html>
'''


## salida

In [29]:
# Guardar el archivo HTML
usuario = os.getlogin()
destino_html = fr"C:\Users\{usuario}\Downloads\nuevo-f\formulario-ang\index.html"

try:
    with open(destino_html, "w", encoding="utf-8") as file:
        file.write(html_content)
    
   
    
    webbrowser.open(destino_html)
    
except Exception as e:
    
    destino_actual = os.path.join(os.getcwd(), "cuestionario_equipamiento_imss.html")
    with open(destino_actual, "w", encoding="utf-8") as file:
        file.write(html_content)
 
    webbrowser.open(destino_actual)